# Bài 3: Bài Tập Thực Hành - Kỹ Thuật Xử Lý Dữ Liệu

**Hướng dẫn:**
- Hoàn thành các câu hỏi và bài tập dưới đây để củng cố kiến thức về xử lý dữ liệu cho LightGBM.
- Cố gắng tự trả lời trước khi xem đáp án.
- Chạy các ô code để kiểm tra kết quả của bạn.

---

### Phần 1: Câu hỏi lý thuyết

**Câu 1:** So sánh hai phương pháp xử lý biến categorical là One-Hot Encoding và Label Encoding. Nêu rõ ưu và nhược điểm của từng phương pháp, đặc biệt là trong bối cảnh các mô hình dựa trên cây quyết định.

**Câu 2:** LightGBM có một cơ chế xử lý biến categorical rất hiệu quả. Giải thích cách nó hoạt động. Tại sao cơ chế này thường tốt hơn One-Hot Encoding, đặc biệt là với các biến có độ phân giải cao (high cardinality)?

**Câu 3:** Để sử dụng tính năng xử lý categorical tự nhiên của LightGBM, bạn cần làm gì với DataFrame của mình trước khi đưa vào mô hình?

**Câu 4:** Các mô hình cây như LightGBM xử lý các giá trị thiếu (missing values, `NaN`) như thế nào? Tại sao việc này lại là một lợi thế so với các mô hình khác như Logistic Regression hay SVM?

**Câu 5:** Giả sử bạn có một cột categorical chứa các giá trị thiếu. Bạn nên xử lý các giá trị thiếu này như thế nào *trước khi* sử dụng tính năng xử lý categorical tự nhiên của LightGBM? Tại sao?

---

### Phần 2: Bài tập thực hành

**Bối cảnh:** Chúng ta sẽ làm việc với bộ dữ liệu Titanic. Mục tiêu là dự đoán một hành khách có sống sót (`Survived`) hay không. Bộ dữ liệu này chứa cả biến số, biến categorical và các giá trị thiếu, rất phù hợp để thực hành.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# Tải dữ liệu
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df_titanic = pd.read_csv(url)

df_titanic.info()

**Phân tích sơ bộ:**
- `Age`, `Cabin`, `Embarked` có giá trị thiếu.
- `Name`, `Sex`, `Ticket`, `Cabin`, `Embarked` là các biến categorical.
- `PassengerId`, `Name`, `Ticket` có thể không hữu ích cho mô hình và nên được loại bỏ.

**Bài tập 6: Tiền xử lý dữ liệu**

Hãy viết code để thực hiện các bước sau:
1.  Tạo một DataFrame mới `df_processed` từ `df_titanic`.
2.  Loại bỏ các cột `PassengerId`, `Name`, `Ticket`, `Cabin`. (Cột `Cabin` có quá nhiều giá trị thiếu nên chúng ta bỏ qua trong bài tập này).
3.  Xác định các cột categorical còn lại (`Sex`, `Embarked`).
4.  Đối với mỗi cột categorical này, hãy điền các giá trị thiếu (`NaN`) bằng chuỗi `'Unknown'`.
5.  Chuyển đổi kiểu dữ liệu của các cột categorical này thành `'category'`.

In [ ]:
# Viết code của bạn vào đây
df_processed = df_titanic.copy()

# 1. Loại bỏ cột
df_processed = df_processed.drop(columns=[...])

# 2. Xác định cột categorical
categorical_cols = [...] 

# 3 & 4. Điền giá trị thiếu và chuyển đổi kiểu dữ liệu
for col in categorical_cols:
    df_processed[col] = df_processed[col].fillna('Unknown')
    df_processed[col] = df_processed[col].astype(...)

# Cột Age là biến số, LightGBM có thể tự xử lý NaN, nên chúng ta không cần điền

df_processed.info()

**Bài tập 7: Huấn luyện và Đánh giá mô hình**

1.  Tách dữ liệu đã xử lý thành `X` (features) và `y` (`Survived`).
2.  Chia thành tập train và test (tỷ lệ 80/20, `random_state=42`).
3.  Khởi tạo và huấn luyện một mô hình `lgb.LGBMClassifier` với `random_state=42`.
4.  Dự đoán trên tập test và in ra `accuracy` và `roc_auc_score`.

In [ ]:
# Viết code của bạn vào đây
X = df_processed.drop('Survived', axis=1)
y = df_processed['Survived']

X_train, X_test, y_train, y_test = ...

lgbm_titanic = ...
lgbm_titanic.fit(X_train, y_train)

y_pred = lgbm_titanic.predict(X_test)
y_proba = lgbm_titanic.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

**Câu hỏi tư duy:** Bạn có nhận thấy chúng ta đã không cần phải điền giá trị thiếu cho cột `Age` không? Tại sao mô hình vẫn chạy được và cho kết quả tốt? Điều này cho thấy lợi ích gì của LightGBM?

---

## Đáp Án

### Phần 1: Lý thuyết

**Câu 1:**
- **Label Encoding:**
  - *Ưu điểm:* Đơn giản, không làm tăng số chiều dữ liệu.
  - *Nhược điểm:* Tạo ra một thứ tự giả tạo giữa các giá trị (ví dụ: 2 > 1 > 0), điều mà các mô hình cây có thể diễn giải sai là một tín hiệu có ý nghĩa, dẫn đến hiệu suất kém.
- **One-Hot Encoding:**
  - *Ưu điểm:* Không tạo ra thứ tự giả, thể hiện rõ ràng sự khác biệt giữa các category.
  - *Nhược điểm:* Với các biến có nhiều giá trị (high cardinality), nó tạo ra rất nhiều cột mới, làm tăng mạnh số chiều dữ liệu, gây tốn bộ nhớ, làm chậm quá trình huấn luyện và có thể làm giảm hiệu suất của mô hình cây (vì mỗi cây chỉ có thể chọn một phần nhỏ các feature).

**Câu 2:**
LightGBM xử lý biến categorical bằng cách tìm cách phân chia tối ưu các category thành 2 nhóm, thay vì chia trên một ngưỡng giá trị số. Nó duyệt qua các cách kết hợp các category và chọn cách nào mang lại mức giảm loss (information gain) lớn nhất. 
Cơ chế này tốt hơn OHE vì:
1.  **Không làm tăng số chiều:** Nó không tạo thêm cột mới, giúp tiết kiệm bộ nhớ và tăng tốc độ.
2.  **Tìm ra mối quan hệ phức tạp:** Nó có thể nhóm các category có ý nghĩa tương tự lại với nhau, một điều mà OHE không làm được (OHE coi mọi category đều trực giao/độc lập với nhau).

**Câu 3:**
Để sử dụng tính năng này, bạn cần chuyển đổi kiểu dữ liệu (dtype) của các cột categorical trong Pandas DataFrame thành `'category'`. LightGBM sẽ tự động phát hiện các cột có kiểu dữ liệu này và áp dụng thuật toán xử lý categorical đặc biệt của nó.

**Câu 4:**
Khi gặp giá trị thiếu (`NaN`) trong một cột, thuật toán xây dựng cây sẽ không bỏ qua hay báo lỗi. Thay vào đó, nó sẽ thử đưa tất cả các hàng có giá trị thiếu vào nhánh trái và tính gain, sau đó thử đưa tất cả vào nhánh phải và tính gain. Nó sẽ chọn hướng đi nào (trái hoặc phải) mang lại gain cao hơn làm hướng đi mặc định cho các giá trị thiếu tại nút chia đó. 
Đây là một lợi thế lớn vì các mô hình khác như Logistic Regression hay SVM yêu cầu tất cả dữ liệu phải đầy đủ. Điều này buộc người dùng phải thực hiện bước tiền xử lý là điền giá trị thiếu (imputation), vốn có thể đưa vào các giả định không chính xác về dữ liệu.

**Câu 5:**
Bạn nên điền các giá trị thiếu trong cột categorical bằng một giá trị đại diện, ví dụ như một chuỗi mới như `'Unknown'` hoặc `'Missing'`. Sau đó, bạn mới chuyển đổi toàn bộ cột sang kiểu `'category'`. 
Lý do là để LightGBM có thể coi việc "thiếu thông tin" như một category riêng biệt. Mô hình có thể học được rằng việc một giá trị bị thiếu trong cột `Embarked` (ví dụ) là một tín hiệu quan trọng và có thể sử dụng nó để đưa ra quyết định phân loại.

### Phần 2: Thực hành (Code đáp án)

In [ ]:
# Bài tập 6: Tiền xử lý dữ liệu
df_processed = df_titanic.copy()

# 1. Loại bỏ cột
df_processed = df_processed.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# 2. Xác định cột categorical
categorical_cols = ['Sex', 'Embarked']

# 3 & 4. Điền giá trị thiếu và chuyển đổi kiểu dữ liệu
for col in categorical_cols:
    df_processed[col] = df_processed[col].fillna('Unknown')
    df_processed[col] = df_processed[col].astype('category')

print("Dữ liệu sau khi xử lý:")
df_processed.info()

In [ ]:
# Bài tập 7: Huấn luyện và Đánh giá mô hình
X = df_processed.drop('Survived', axis=1)
y = df_processed['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lgbm_titanic = lgb.LGBMClassifier(random_state=42)
lgbm_titanic.fit(X_train, y_train)

y_pred = lgbm_titanic.predict(X_test)
y_proba = lgbm_titanic.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.4f}")

**Câu trả lời cho câu hỏi tư duy:**
Chúng ta không cần điền giá trị thiếu cho cột `Age` vì nó là một biến số (numerical), và LightGBM có khả năng xử lý các giá trị `NaN` trong biến số một cách tự nhiên. Mô hình đã tự động học được cách tốt nhất để xử lý các hành khách bị thiếu tuổi - có thể bằng cách gộp họ vào nhóm những người trẻ tuổi hoặc người lớn tuổi, tùy thuộc vào cách nào giúp phân loại tốt hơn.

3
4
Đây là một lợi ích cực kỳ lớn, giúp tiết kiệm thời gian và công sức trong bước tiền xử lý dữ liệu, một bước thường rất tốn kém trong các dự án khoa học dữ liệu.